# 情報数学Ⅲ 第10回

In [ ]:
# 必要なデータファイルを取得
import requests

base_url = "https://raw.githubusercontent.com/logics-of-blue/book-python-stats-2nd/refs/heads/main/book-data/"
filenames = [
    "8-1-1-beer.csv"
]

for filename in filenames:
    url = base_url + filename
    print(f"Downloading {filename}...")
    response = requests.get(url)
    if response.status_code == 200:
        with open(filename, "wb") as f:
            f.write(response.content)
    else:
        print(f"Failed to download {filename}: {response.status_code}")


# 第8部　正規線形モデル

## 1章　連続型の説明変数を1つ持つモデル

### 実装：分析の準備

In [ ]:
# 数値計算に使うライブラリ
import numpy as np
import pandas as pd
from scipy import stats
# 表示桁数の設定
pd.set_option('display.precision', 3)
np.set_printoptions(precision=3)

# グラフを描画するライブラリ
from matplotlib import pyplot as plt
import seaborn as sns
sns.set()

# 統計モデルを推定するライブラリ
import statsmodels.formula.api as smf
import statsmodels.api as sm

In [ ]:
# 表示設定(書籍本文のレイアウトと合わせるためであり、必須ではありません)
np.set_printoptions(linewidth=60)
pd.set_option('display.width', 60)

from matplotlib.pylab import rcParams
rcParams['figure.figsize'] = 8, 4

### 実装：データの読み込みと図示

In [ ]:
# データの読み込み
beer = pd.read_csv('8-1-1-beer.csv')
print(beer.head(n=3))

In [ ]:
# 図示
sns.scatterplot(x='temperature', y='beer', 
                data=beer, color='black')

### 実装：係数の推定

In [ ]:
# データの準備
x = beer['temperature']
y = beer['beer']

In [ ]:
# 分散共分散行列
cov_mat = np.cov(x, y, ddof=0)
cov_mat

In [ ]:
# 係数の推定

# 平均値
x_bar = np.mean(x)
y_bar = np.mean(y)

# 共分散と分散
cov_xy =  cov_mat[0, 1]
s2_x = cov_mat[0, 0]

# 係数の推定
beta_1 = cov_xy / s2_x
beta_0 = y_bar - beta_1 * x_bar

print('切片      : ', round(beta_0, 3))
print('気温の係数: ', round(beta_1, 3))

### 実装：statsmodelsによるモデル化

In [ ]:
# モデルの構築
lm_model = smf.ols(formula='beer ~ temperature', 
                   data=beer).fit()

### 実装：推定結果の表示と係数の検定

In [ ]:
# 推定結果の表示
lm_model.summary()

### 実装：AICによるモデル選択

In [ ]:
# NULLモデル
null_model = smf.ols(formula='beer ~ 1', data=beer).fit()

In [ ]:
# NULLモデルのAIC
round(null_model.aic, 3)

In [ ]:
# 説明変数入りのモデルのAIC
round(lm_model.aic, 3)

In [ ]:
# 対数尤度
round(lm_model.llf, 3)

In [ ]:
# 説明変数の数
lm_model.df_model

In [ ]:
# AIC
round(-2 * (lm_model.llf - (lm_model.df_model + 1)), 3)

### 実装：単回帰による予測

#### 当てはめ値

In [ ]:
# 当てはめ値
lm_model.predict()

In [ ]:
# 参考：当てはめ値を取得する別の方法(書籍には載っていないコードです)
lm_model.fittedvalues

#### 気温が0度のときの予測値

In [ ]:
# 予測
lm_model.predict(pd.DataFrame({'temperature':[0]}))

In [ ]:
# 気温0どの時の予測値は切片に等しい
lm_model.params

#### 気温が20度のときの予測値

In [ ]:
# 予測
lm_model.predict(pd.DataFrame({'temperature':[20]}))

In [ ]:
# predict関数を使わないで予測
beta0 = lm_model.params[0]
beta1 = lm_model.params[1]
temperature = 20

round(beta0 + beta1 * temperature, 3)

### 実装：信頼区間・予測区間

In [ ]:
# 当てはめ結果の信頼区間と予測区間を得る
pred_interval = lm_model.get_prediction(
    pd.DataFrame({'temperature':[20]}))
pred_frame = pred_interval.summary_frame(alpha=0.05)
print(pred_frame)

### 実装：seabornによる回帰直線の図示

In [ ]:
sns.lmplot(x='temperature', y='beer', data=beer,
           scatter_kws={'color': 'black'},
           line_kws   ={'color': 'black'},
           ci=None, height=4, aspect=2)

In [ ]:
# 参考：axis-level関数(書籍には載っていないコードです)
sns.regplot(x='temperature', y='beer', data=beer,
           scatter_kws={'color': 'black'},
           line_kws   ={'color': 'black'},
           ci=None)

### 実装：信頼区間と予測区間の図示

In [ ]:
# 当てはめ結果の信頼区間と予測区間を得る
pred_all = lm_model.get_prediction()
pred_frame_all = pred_all.summary_frame(alpha=0.05)

In [ ]:
# 説明変数を付け加える
pred_graph = pd.concat(
    [beer.temperature, pred_frame_all], axis = 1)

# 図示のためにソートする
pred_graph = pred_graph.sort_values("temperature")

In [ ]:
# 参考：グラフ描画用のデータにおける、最初の3行(書籍には載っていないコードです)
pred_graph.head(3)

In [ ]:
# 散布図
sns.scatterplot(x='temperature', y='beer', 
                data=beer, color='black')

# 回帰直線
sns.lineplot(x='temperature', y='mean', 
             data=pred_graph, color='black')

# 信頼区間
sns.lineplot(x='temperature', y='mean_ci_lower', 
             data=pred_graph, color='black', 
             linestyle='dashed')
sns.lineplot(x='temperature', y='mean_ci_upper', 
             data=pred_graph, color='black', 
             linestyle='dashed')

# 予測区間
sns.lineplot(x='temperature', y='obs_ci_lower', 
             data=pred_graph, color='black', 
             linestyle='dotted')
sns.lineplot(x='temperature', y='obs_ci_upper', 
             data=pred_graph, color='black', 
             linestyle='dotted')